# UniVerse Creator Engine
Runtime de processamento do UniVerse Creator. O aplicativo e suas telas não são alterados por este notebook.

Este notebook usa Google Drive para armazenar identidades e resultados e ComfyUI como motor de geração. É necessário selecionar um runtime com GPU e colocar um checkpoint compatível em `UniVerseCreator/models/checkpoints`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/UniVerseCreator')
for name in ['identities','generated','loras','models/checkpoints','models/controlnet','models/animatediff']:
    (ROOT/name).mkdir(parents=True,exist_ok=True)
print('Drive pronto:', ROOT)

In [ ]:
!git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt fastapi uvicorn requests python-multipart
!mkdir -p /content/ComfyUI/custom_nodes
!git clone https://github.com/Fannovel16/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
!git clone https://github.com/Fannovel16/comfyui_controlnet_aux.git /content/ComfyUI/custom_nodes/comfyui_controlnet_aux
!git clone https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git /content/ComfyUI/custom_nodes/ComfyUI-AnimateDiff-Evolved
!ln -sfn /content/drive/MyDrive/UniVerseCreator/models/checkpoints /content/ComfyUI/models/checkpoints
!ln -sfn /content/drive/MyDrive/UniVerseCreator/models/controlnet /content/ComfyUI/models/controlnet
!ln -sfn /content/drive/MyDrive/UniVerseCreator/loras /content/ComfyUI/models/loras
print('ComfyUI e nós de extensão instalados. Adicione um checkpoint SDXL e os modelos ControlNet em UniVerseCreator/models.')

## Serviço de API
A célula abaixo inicia o ComfyUI e a API usada pelo Cloudflare. Copie a URL HTTPS exibida para configurar `COLAB_MODULE1_URL` com `/create_identity` e `COLAB_MODULE2_URL` com `/generate_image`.

In [ ]:
import os, subprocess, threading, time, requests
from google.colab import output
os.environ['COMFYUI_URL']='http://127.0.0.1:8188'
comfy=subprocess.Popen(['python','main.py','--listen','127.0.0.1','--port','8188'],cwd='/content/ComfyUI',stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
time.sleep(20)
print('ComfyUI:', requests.get('http://127.0.0.1:8188/system_stats',timeout=20).status_code)
%cd /content
!wget -q -O /content/uvc_engine.py https://raw.githubusercontent.com/melpfintner/universe-creator/main/colab/uvc_engine.py
def run_api():
    import uvicorn
    uvicorn.run('uvc_engine:app',host='0.0.0.0',port=8000,log_level='info')
threading.Thread(target=run_api,daemon=True).start()
time.sleep(3)
public_url=output.eval_js('google.colab.kernel.proxyPort(8000)')
print('URL HTTPS DO COLAB:', public_url)
print('COLAB_MODULE1_URL =', public_url.rstrip('/')+'/create_identity')
print('COLAB_MODULE2_URL =', public_url.rstrip('/')+'/generate_image')
print('Health:', requests.get(public_url.rstrip('/')+'/health').text)

## Observações técnicas
- A geração real exige um checkpoint SDXL no Google Drive. Sem pesos, o ComfyUI não consegue gerar imagens.
- ControlNet precisa dos modelos de controle correspondentes e dos nós auxiliares instalados no ComfyUI. A rota base não finge que ControlNet ou AnimateDiff estão ativos.
- Kohya_ss exige um conjunto de fotos da identidade, captions e GPU adequada. Não inicie treinamento com apenas uma foto.
- AnimateDiff exige motion module compatível e workflow ComfyUI específico; a rota de vídeo só deve ser habilitada após instalar e validar esses componentes.
- O endereço público do Colab muda quando o runtime reinicia. Atualize as duas variáveis do Worker quando isso ocorrer.